In [36]:
import os
import sys
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

G = 0.02
DELTA = 0.03
TARGET = "growth_10y_avg"
FEATURES_THEORY = ["ln_y0", "ln_s", "ln_ngd"]
FEATURES_H = ["ln_y0", "ln_s", "ln_ngd", "ln_h"]

def evaluate(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    return {
        "model": name,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_test, preds),
        "R2_oos": r2_score(y_test, preds)
    }

def run_baselines(X_train, y_train, X_test, y_test, suffix=""):
    results = []

    results.append(evaluate(
        DummyRegressor(strategy="mean"),
        X_train, y_train, X_test, y_test,
        f"Mean{suffix}"
    ))

    results.append(evaluate(
        LinearRegression(),
        X_train, y_train, X_test, y_test,
        f"Linear{suffix}"
    ))

    results.append(evaluate(
        Ridge(alpha=1.0),
        X_train, y_train, X_test, y_test,
        f"Ridge{suffix}"
    ))

    results.append(evaluate(
        Lasso(alpha=0.001),
        X_train, y_train, X_test, y_test,
        f"Lasso{suffix}"
    ))

    return pd.DataFrame(results).sort_values("RMSE")

def add_growth_target_and_solow_features(df, g=G, delta=DELTA):
    df = df.sort_values(["country", "year"]).reset_index(drop=True).copy()

    df["gdp_pc_t10"] = df.groupby("country")["gdp_per_capita"].shift(-1)
    df["growth_10y_avg"] = (
        np.log(df["gdp_pc_t10"]) - np.log(df["gdp_per_capita"])
    ) / 10

    df["n_g_delta"] = df["population_growth"] / 100 + g + delta

    return df

def prepare_theory_dataset(df):
    df = df[
        (df["gdp_per_capita"] > 0) &
        (df["investment_rate"] > 0) &
        (df["n_g_delta"] > 0)
    ].copy()

    df["ln_y0"] = np.log(df["gdp_per_capita"])
    df["ln_s"] = np.log(df["investment_rate"] / 100)
    df["ln_ngd"] = np.log(df["n_g_delta"])

    ml_df = df.dropna(subset=[
        "growth_10y_avg", "ln_y0", "ln_s", "ln_ngd"
    ]).copy()

    return ml_df

def prepare_human_capital_dataset(df):
    df = df[
        (df["gdp_per_capita"] > 0) &
        (df["investment_rate"] > 0) &
        (df["school_enrollment"] > 0) &
        (df["n_g_delta"] > 0)
    ].copy()

    df["ln_y0"] = np.log(df["gdp_per_capita"])
    df["ln_s"] = np.log(df["investment_rate"] / 100)
    df["ln_ngd"] = np.log(df["n_g_delta"])
    df["ln_h"] = np.log(df["school_enrollment"])

    ml_df = df.dropna(subset=[
        "growth_10y_avg", "ln_y0", "ln_s", "ln_ngd", "ln_h"
    ]).copy()

    return ml_df

def time_split(df, feature_cols, target=TARGET):
    train_df = df[df["year"].isin([1990, 2000])].copy()
    test_df = df[df["year"] == 2010].copy()

    X_train = train_df[feature_cols]
    y_train = train_df[target]

    X_test = test_df[feature_cols]
    y_test = test_df[target]

    return train_df, test_df, X_train, y_train, X_test, y_test

df_theory_raw = pd.read_csv("../data/processed/wdi_solow_tidy.csv")
df_theory_raw = add_growth_target_and_solow_features(df_theory_raw)

ml_df = prepare_theory_dataset(df_theory_raw)

print("Theory-only dataset shape:", ml_df.shape)
print(ml_df["year"].value_counts().sort_index())
ml_df.head()

train_df, test_df, X_train, y_train, X_test, y_test = time_split(
    ml_df, FEATURES_THEORY
)

print("Train years:")
print(train_df["year"].value_counts().sort_index())

print("\nTest years:")
print(test_df["year"].value_counts().sort_index())

print("\nX shapes:")
print(X_train.shape, X_test.shape)

results_df = run_baselines(
    X_train, y_train, X_test, y_test,
    suffix=" (theory)"
)

results_df["feature_set"] = "theory_only"
results_df

df_h_raw = pd.read_csv("../data/raw/wdi_solow_raw_extended.csv")

df_long = df_h_raw.melt(
    id_vars=["country", "series"],
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].str.replace("YR", "", regex=False).astype(int)

indicator_map = {
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "NE.GDI.TOTL.ZS": "investment_rate",
    "SP.POP.GROW": "population_growth",
    "SE.SEC.ENRR": "school_enrollment"
}

df_long["indicator"] = df_long["series"].map(indicator_map)

df_h_tidy = df_long.pivot_table(
    index=["country", "year"],
    columns="indicator",
    values="value"
).reset_index()

df_h_tidy.columns.name = None

df_h_tidy.to_csv("../data/processed/wdi_solow_tidy_extended.csv", index=False)

df_h_tidy = add_growth_target_and_solow_features(df_h_tidy)
ml_df_h = prepare_human_capital_dataset(df_h_tidy)

print("Theory + H dataset shape:", ml_df_h.shape)
print(ml_df_h["year"].value_counts().sort_index())
ml_df_h.head()

train_df_h, test_df_h, X_train_h, y_train_h, X_test_h, y_test_h = time_split(
    ml_df_h, FEATURES_H
)

print("Train years:")
print(train_df_h["year"].value_counts().sort_index())

print("\nTest years:")
print(test_df_h["year"].value_counts().sort_index())

print("\nX shapes:")
print(X_train_h.shape, X_test_h.shape)

results_h_df = run_baselines(
    X_train_h, y_train_h, X_test_h, y_test_h,
    suffix=" + H"
)

results_h_df["feature_set"] = "theory_plus_h"
results_h_df

comparison_df = pd.concat([results_df, results_h_df], ignore_index=True)
comparison_df = comparison_df.sort_values(["feature_set", "RMSE"])

comparison_df

# Countries available in both test sets
common_countries = set(test_df["country"]).intersection(set(test_df_h["country"]))

print("Number of common countries:", len(common_countries))

# Align THEORY dataset
train_aligned = train_df[train_df["country"].isin(common_countries)].copy()
test_aligned  = test_df[test_df["country"].isin(common_countries)].copy()

# Align THEORY + H dataset
train_h_aligned = train_df_h[train_df_h["country"].isin(common_countries)].copy()
test_h_aligned  = test_df_h[test_df_h["country"].isin(common_countries)].copy()

# THEORY
X_train_aligned = train_aligned[FEATURES_THEORY]
y_train_aligned = train_aligned[TARGET]

X_test_aligned  = test_aligned[FEATURES_THEORY]
y_test_aligned  = test_aligned[TARGET]

# THEORY + H
X_train_h_aligned = train_h_aligned[FEATURES_H]
y_train_h_aligned = train_h_aligned[TARGET]

X_test_h_aligned  = test_h_aligned[FEATURES_H]
y_test_h_aligned  = test_h_aligned[TARGET]

results_aligned = run_baselines(
    X_train_aligned, y_train_aligned,
    X_test_aligned, y_test_aligned,
    suffix=" (aligned theory)"
)

results_h_aligned = run_baselines(
    X_train_h_aligned, y_train_h_aligned,
    X_test_h_aligned, y_test_h_aligned,
    suffix=" (aligned + H)"
)

results_aligned["feature_set"] = "theory_only_aligned"
results_h_aligned["feature_set"] = "theory_plus_h_aligned"

comparison_aligned = pd.concat([results_aligned, results_h_aligned], ignore_index=True)
comparison_aligned = comparison_aligned.sort_values(["feature_set", "RMSE"])

comparison_aligned

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

results_ml = []

results_ml.append(evaluate(
    RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42),
    X_train_h_aligned, y_train_h_aligned,
    X_test_h_aligned, y_test_h_aligned,
    "Random Forest (aligned)"
))

results_ml.append(evaluate(
    GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=42),
    X_train_h_aligned, y_train_h_aligned,
    X_test_h_aligned, y_test_h_aligned,
    "Gradient Boosting (aligned)"
))

results_ml_df = pd.DataFrame(results_ml)
results_ml_df

final_results = pd.concat([
    comparison_aligned,
    results_ml_df
], ignore_index=True)

final_results.to_csv("../reports/tables/final_model_comparison.csv", index=False)

Theory-only dataset shape: (571, 11)
year
1990    166
2000    196
2010    209
Name: count, dtype: int64
Train years:
year
1990    166
2000    196
Name: count, dtype: int64

Test years:
year
2010    209
Name: count, dtype: int64

X shapes:
(362, 3) (209, 3)
Theory + H dataset shape: (376, 13)
year
1990     67
2000    146
2010    163
Name: count, dtype: int64
Train years:
year
1990     67
2000    146
Name: count, dtype: int64

Test years:
year
2010    163
Name: count, dtype: int64

X shapes:
(213, 4) (163, 4)
Number of common countries: 163


,model,RMSE,MAE,R2_oos
0,Random Forest (aligned),0.025567,0.017628,-0.297863
1,Gradient Boosting (aligned),0.027899,0.019486,-0.545362
